# Mimariler Arası Tahmin Uyuşmazlığı ve Eskalasyon Analizi
**Chest X-Ray Tiered Classification · Tier 1 ve Tier 2 Çelişki İncelemesi**

Bu defter, kademeli yönlendirme mimarimizin kalbi olan **Dynamic Router (Dinamik Yönlendirici)** mekanizmasını derinlemesine inceler.

**Ana Amaç:** Tier 1 (hafif MobileNetV2) modelinin tek başına karar veremeyip, belirsizlik (Uncertainty) limitlerini aşarak Tier 2 (ağır Ark+ Swin) modeline devrettiği (escalation) çelişkili vakaları incelemek, eskalasyon kararlarının doğruluğunu ve maliyet/performans dengesini ortaya koymaktır.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# REAL Tier 1 / Tier 2 probabilities, MC-dropout uncertainty and routing decisions.
_PRED = next((p for p in ["../outputs/results/tiered_predictions.csv",
                          "outputs/results/tiered_predictions.csv"]
              if os.path.exists(p)), None)
if _PRED is None:
    raise FileNotFoundError(
        "tiered_predictions.csv not found. Generate REAL predictions first:\n"
        "    python scripts/export_predictions.py --tier2-backbone ark_plus\n"
        "(run on Colab after the models are trained/restored)."
    )
df = pd.read_csv(_PRED)
df = df.rename(columns={"tier1_prob": "t1_prob", "tier2_prob": "t2_prob"})
print(f"Loaded {len(df)} REAL predictions. Escalated to Tier 2: {int(df['escalated'].sum())}.")


### Eskalasyon Kararlarının Güvenilirlik Dağılım Grafiği
Aşağıdaki saçılım grafiği (scatter plot), Tier 1 tahmin olasılıklarına karşılık gelen belirsizlikleri göstermekte ve hangi vakaların doğru şekilde üst kademe uzman modele (Tier 2) paslandığını renklendirmektedir.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df[~df['escalated']]['t1_prob'], df[~df['escalated']]['t1_uncertainty'],
            color='green', alpha=0.6, label='Resolved by Tier 1 (fast / cheap)')
plt.scatter(df[df['escalated']]['t1_prob'], df[df['escalated']]['t1_uncertainty'],
            color='orange', alpha=0.7, label='Escalated to Tier 2 (accuracy-focused)')
# Escalation rule: Tier 1 confidence below threshold, i.e. probability inside this band.
for _b in (0.25, 0.75):
    plt.axvline(x=_b, color='red', linestyle='--', alpha=0.7)
plt.xlabel('Tier 1 Pneumothorax probability')
plt.ylabel('Tier 1 MC-dropout uncertainty')
plt.title('Dynamic Router Escalation & Disagreement (real test predictions)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()


### Eskalasyon Kazanç Özeti

In [ ]:
escalated_count = df['escalated'].sum()
pct_saved = (1 - escalated_count / len(df)) * 100
print(f"Toplam Teşhis Edilen Örnek: {len(df)}")
print(f"Tier 1 Tarafından Çözülen (Eskale edilmeyen): {len(df) - escalated_count} (%{pct_saved:.1f})")
print(f"Tier 2\'ye Paslanan (Eskale edilen): {escalated_count} (%{100 - pct_saved:.1f})")

# Tier 1 doğruluğu vs Tiered Sistem doğruluğu
t1_correct = ((df['t1_prob'] >= 0.5).astype(int) == df['y_true']).mean() * 100
df['tiered_prob'] = np.where(df['escalated'], df['t2_prob'], df['t1_prob'])
tiered_correct = ((df['tiered_prob'] >= 0.5).astype(int) == df['y_true']).mean() * 100
print(f"\nTier 1 Tek Başına Doğruluk Oranı: %{t1_correct:.1f}")
print(f"Kademeli Sistem (Tiered) Birleşik Doğruluk Oranı: %{tiered_correct:.1f}")
print(f"İyileşme (Performans Artışı): +%{tiered_correct - t1_correct:.1f}")